In [1]:
# load snapshot data
import pandas as pd

snapshot = pd.read_csv(
    "../feature_store/student_snapshot.csv"
)  
behaviour = pd.read_csv(
    "../feature_store/behaviour_model_snapshot.csv"
)
academic= pd.read_csv(
    "../feature_store/academic_model_snapshot_v2.csv"
) 

In [2]:
# target 
snapshot["grade_code_mode"].value_counts()

grade_code_mode
N            73
P            23
C            19
D            16
S            14
SA           13
HD            9
DX            3
No Result     1
Name: count, dtype: int64

In [3]:
# check how many top_material_type categories we have
snapshot["top_material_type"].value_counts()

top_material_type
Course Page         123
Assignment           35
Lecture Material     11
No Activity           2
Name: count, dtype: int64

In [4]:
# encoding catergorical features - one hot encoding
behaviour = pd.get_dummies(
    behaviour,
    columns=["top_material_type"],
    drop_first=False
)
behaviour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   student_key                         167 non-null    int64  
 1   course_key                          167 non-null    int64  
 2   snapshot_date                       167 non-null    object 
 3   total_events                        167 non-null    int64  
 4   active_days                         167 non-null    int64  
 5   unique_materials                    167 non-null    int64  
 6   after_hours_events                  167 non-null    int64  
 7   weekend_events                      167 non-null    int64  
 8   avg_events_per_active_day           167 non-null    float64
 9   events_per_week                     167 non-null    float64
 10  weekend_ratio                       167 non-null    float64
 11  after_hours_ratio                   167 non-n

In [5]:
print(behaviour.columns.tolist())
print(academic.columns.tolist())

['student_key', 'course_key', 'snapshot_date', 'total_events', 'active_days', 'unique_materials', 'after_hours_events', 'weekend_events', 'avg_events_per_active_day', 'events_per_week', 'weekend_ratio', 'after_hours_ratio', 'lecture_material_views', 'assignment_views', 'course_page_views', 'forum_views', 'quiz_views', 'no_activity_flag', 'risk_label', 'top_material_type_Assignment', 'top_material_type_Course Page', 'top_material_type_Lecture Material', 'top_material_type_No Activity']
['student_key', 'course_key', 'avg_score_so_far', 'max_score_so_far', 'min_score_so_far', 'pass_assessment_count', 'latest_assessment_score', 'fail_assessment_count', 'early_exercise_avg', 'late_exercise_avg', 'assignment_avg_score', 'score_trend', 'snapshot_date', 'risk_label']


In [6]:
if "top_material_type_No Activity" in behaviour.columns:
    print(behaviour["top_material_type_No Activity"].sum())

if "top_material_type_No Activity" in academic.columns:
    print(academic["top_material_type_No Activity"].sum())

2


In [7]:
snapshot[
    snapshot["top_material_type"] == "No Activity"
][[
    "student_key",
    "total_events",
    "active_days",
    "grade_code_mode"
]]

,student_key,total_events,active_days,grade_code_mode
169,170,0,0,N
170,171,0,0,N


In [8]:
behaviour["top_material_type_No Activity"].value_counts()

top_material_type_No Activity
False    165
True       2
Name: count, dtype: int64

In [9]:
snapshot["top_material_type"].value_counts()

top_material_type
Course Page         123
Assignment           35
Lecture Material     11
No Activity           2
Name: count, dtype: int64

In [10]:
snapshot_with_risk = snapshot.merge(
    behaviour[["student_key", "course_key", "snapshot_date", "risk_label"]],
    on=["student_key", "course_key", "snapshot_date"],
    how="left"
)

pd.crosstab(
    snapshot_with_risk["top_material_type"],
    snapshot_with_risk["risk_label"],
    normalize="index"
)

risk_label,0.0,1.0
top_material_type,,
Assignment,0.500000,0.500000
Course Page,0.375000,0.625000
Lecture Material,0.454545,0.545455
No Activity,0.000000,1.000000


In [11]:
# behaviour model dataset preparation
# train-val-test split
from sklearn.model_selection import train_test_split

X_behaviour = behaviour.drop(
    columns=[
        "student_key",
        "course_key",
        "snapshot_date",
        "risk_label",
        "quiz_views",  # zero variance
        "avg_events_per_active_day",  # 1 corr with events_per_week
        "top_material_type_Assignment",  # low importance
        "top_material_type_Course Page",  # low importance
        "top_material_type_Lecture Material",  # low importance
        "top_material_type_No Activity"  # low importance
    ],
    errors="ignore"
)

y_behaviour = behaviour["risk_label"]

X_beh_train, X_beh_test, y_beh_train, y_beh_test = train_test_split(
    X_behaviour,
    y_behaviour,
    test_size=0.2,
    random_state=42,
    stratify=y_behaviour
)

# validation split (25% of train = 20% of original)
X_beh_train, X_beh_val, y_beh_train, y_beh_val = train_test_split(
    X_beh_train,
    y_beh_train,
    test_size=0.25,
    random_state=42,
    stratify=y_beh_train
)

In [12]:
# academic train-test split
X_academic = academic.drop(
    columns=[
        "student_key",
        "course_key",
        "snapshot_date",
        "risk_label",
        "assessment_completion_rate"
    ],
    errors="ignore"
)

y_academic = academic["risk_label"]

X_aca_train, X_aca_test, y_aca_train, y_aca_test = train_test_split(
    X_academic,
    y_academic,
    test_size=0.2,
    random_state=42,
    stratify=y_academic
)

# validation split (25% of train = 20% of original)
X_aca_train, X_aca_val, y_aca_train, y_aca_val = train_test_split(
    X_aca_train,
    y_aca_train,
    test_size=0.25,
    random_state=42,
    stratify=y_aca_train
)

In [13]:
print(X_beh_train.shape, X_beh_val.shape, X_beh_test.shape)
print(X_aca_train.shape, X_aca_val.shape, X_aca_test.shape)
print(y_beh_train.value_counts(normalize=True))
print(y_aca_train.value_counts(normalize=True))

(99, 13) (34, 13) (34, 13)
(99, 10) (34, 10) (34, 10)
risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64
risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64


In [14]:
# save datasets
from pathlib import Path

dataset_dir = Path("../datasets")
dataset_dir.mkdir(exist_ok=True)

# behvaiour 
X_beh_train.to_csv(dataset_dir / "X_beh_train.csv", index=False)
X_beh_val.to_csv(dataset_dir / "X_beh_val.csv", index=False)
X_beh_test.to_csv(dataset_dir / "X_beh_test.csv", index=False)

y_beh_train.to_csv(dataset_dir / "y_beh_train.csv", index=False)
y_beh_val.to_csv(dataset_dir / "y_beh_val.csv", index=False)
y_beh_test.to_csv(dataset_dir / "y_beh_test.csv", index=False)

# academic
X_aca_train.to_csv(dataset_dir / "X_aca_train.csv", index=False)
X_aca_val.to_csv(dataset_dir / "X_aca_val.csv", index=False)
X_aca_test.to_csv(dataset_dir / "X_aca_test.csv", index=False)

y_aca_train.to_csv(dataset_dir / "y_aca_train.csv", index=False)
y_aca_val.to_csv(dataset_dir / "y_aca_val.csv", index=False)
y_aca_test.to_csv(dataset_dir / "y_aca_test.csv", index=False)

print(y_behaviour.value_counts())
print(y_academic.value_counts())

risk_label
1    100
0     67
Name: count, dtype: int64
risk_label
1    100
0     67
Name: count, dtype: int64


In [15]:
print(y_beh_train.value_counts(normalize=True))
print(y_beh_val.value_counts(normalize=True))
print(y_beh_test.value_counts(normalize=True))

risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64


In [16]:
print(y_aca_train.value_counts(normalize=True))
print(y_aca_val.value_counts(normalize=True))
print(y_aca_test.value_counts(normalize=True))

risk_label
1    0.606061
0    0.393939
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64
risk_label
1    0.588235
0    0.411765
Name: proportion, dtype: float64
